In [ ]:
#| default_exp review

# Structural and style review

> Catch structural notebook errors, hidden-code walls, broken cell state, and missing chapter lessons.

A notebook passes review only when it is both structurally sound and genuinely readable by a human. This module combines `chkstyle` with checks for notebook structure, execution order, state safety, and chapter-level literate craft.

In [ ]:
#| export
import ast, builtins, re
from dataclasses import dataclass, field
from pathlib import Path

from fastcore.nbio import read_nb

from nbskill.core import (cell_defs, cell_directives, cell_imports, cell_source, chapters, exported_py_path,
    heading_info, is_eval_false, is_exported, is_hidden, notebook_paths, parse_cell_ast, project_root)
from nbskill.state import cell_dependencies

In [ ]:
from tempfile import TemporaryDirectory
from fastcore.nbio import mk_cell, new_nb, write_nb
from fastcore.test import test_eq, test_fail

## Diagnostic model

Every finding — whether from `nbskill` or `chkstyle` — uses one compact `Diagnostic` record grouped by `category` (`structure`, `state`, `narrative`, or `style`).

In [ ]:
#| export
@dataclass(frozen=True)
class Diagnostic:
    "One actionable notebook finding with stable cell identity and reader-facing hint."
    code: str
    category: str  # 'structure', 'state', 'narrative', or 'style'
    severity: str  # 'error' or 'warning'
    path: str
    message: str
    cell_id: str = ""
    chapter: str = ""
    lineno: int = 0
    hint: str = ""
    source: str = "nbskill"

    def __str__(self) -> str:
        loc = f"{self.path}#{self.cell_id}" if self.cell_id else self.path
        if self.lineno: loc += f":{self.lineno}"
        ch = f" [{self.chapter}]" if self.chapter else ""
        h = f" (hint: {self.hint})" if self.hint else ""
        return f"- [{self.severity}:{self.category}/{self.code}] {loc}{ch}: {self.message}{h}"

@dataclass
class ReviewReport:
    "Structured review results for one or more notebooks."
    path: str
    diagnostics: list[Diagnostic] = field(default_factory=list)
    metrics: dict = field(default_factory=dict)

    @property
    def ok(self) -> bool: return not any(d.severity == "error" for d in self.diagnostics)

    @property
    def counts(self) -> dict[str, int]:
        out = dict(total=len(self.diagnostics), error=0, warning=0, structure=0, state=0, narrative=0, style=0)
        for d in self.diagnostics:
            if d.severity in out: out[d.severity] += 1
            if d.category in out: out[d.category] += 1
        return out

    def __str__(self) -> str:
        c = self.counts
        header = f"Review {self.path}: {c['total']} finding(s) (errors={c['error']}, structure={c['structure']}, state={c['state']}, narrative={c['narrative']}, style={c['style']})"
        if not self.diagnostics: return header + "\nClean."
        return "\n".join([header, *(str(d) for d in self.diagnostics)])

    def _repr_markdown_(self) -> str: return f"```text\n{self}\n```"

A helper constructs diagnostics consistently with normalized relative paths and chapter context.

In [ ]:
#| export
def _rel(nb_path: Path, root: Path) -> str:
    try: return nb_path.resolve().relative_to(root.resolve()).as_posix()
    except ValueError: return str(nb_path)

def _diag(code: str, category: str, path: str, msg: str, *, cell=None, chapter: str = "",
    lineno: int = 0, hint: str = "", severity: str = "warning", source: str = "nbskill") -> Diagnostic:
    cid = str(cell.get("id", "")) if cell is not None else ""
    return Diagnostic(code, category, severity, path, msg, cid, chapter, lineno, hint, source)

def _chapter_map(nb) -> dict[int, str]:
    mapping = {}
    for ch in chapters(nb):
        for i in range(ch.start, ch.end): mapping[i] = ch.title
    return mapping

## Structural checks

Heading hygiene ensures notebooks collapse cleanly in Jupyter and render with proper nbdev titles. Every heading belongs in its own Markdown cell, levels never skip (such as H1 directly to H3), and library notebooks open with an H1 title and `>` subtitle.

In [ ]:
#| export
def _heading_problems(nb, rel: str, ch_map: dict[int, str]) -> list[Diagnostic]:
    problems = []
    has_exp = any("default_exp" in cell_directives(c) for c in nb.cells)
    has_h1 = any((h := heading_info(c)) and h[0] == 1 for c in nb.cells[:6])
    has_sub = any(c.get("cell_type") == "markdown" and any(ln.strip().startswith(">") for ln in cell_source(c).splitlines()) for c in nb.cells[:6])
    if has_exp and (not has_h1 or not has_sub):
        missing = " and ".join([x for x, ok in (("H1 title", has_h1), ("`>` subtitle", has_sub)) if not ok])
        problems.append(_diag("missing-title-header", "structure", rel, f"Notebook is missing {missing}", hint="Add a standalone `# Title` cell and a `> Subtitle` blockquote near the top"))
    prev_level = 0
    for idx, cell in enumerate(nb.cells):
        h = heading_info(cell)
        if not h: continue
        lvl, title = h
        ch = ch_map.get(idx, "")
        non_empty = [ln.strip() for ln in cell_source(cell).splitlines() if ln.strip()]
        if len(non_empty) > 1 and not (lvl == 1 and all(ln.startswith(">") for ln in non_empty[1:])):
            problems.append(_diag("heading-not-standalone", "structure", rel, f"Heading `{title}` shares a cell with body prose", cell=cell, chapter=ch, hint="Put each heading in its own Markdown cell so Jupyter can collapse the section"))
        if (prev_level and lvl > prev_level + 1) or (not prev_level and lvl > 2):
            expected = prev_level + 1 if prev_level else 2
            problems.append(_diag("heading-level-jump", "structure", rel, f"Heading jumps to H{lvl} (`{title}`)", cell=cell, chapter=ch, hint=f"Use H{expected} instead of H{lvl}"))
        prev_level = lvl
    return problems

Code-structure checks catch syntax errors, imports mixed with logic (which break nbdev documentation builds), duplicate imports within the same scope, stale generated `.py` modules, and oversized notebooks.

In [ ]:
#| export
_PY_CELL_ID_RE = re.compile(r"^#\s*%%\s+.+?\.ipynb\s+#([A-Za-z0-9_-]+)", re.MULTILINE)

def _is_import_stmt(node) -> bool: return isinstance(node, (ast.Import, ast.ImportFrom))

def _export_sync_problems(nb_path: Path, nb, rel: str) -> list[Diagnostic]:
    py_path = exported_py_path(nb_path, nb)
    if py_path is None: return []
    if not py_path.is_file():
        return [_diag("exported-py-missing", "structure", rel, f"Generated module `{py_path.name}` is missing", severity="error", hint="Run `nbdev-export`")]
    py_text = py_path.read_text(encoding="utf-8", errors="replace")
    py_ids = set(_PY_CELL_ID_RE.findall(py_text))
    nb_ids = {str(c.get("id", "")) for c in nb.cells if is_exported(c) and c.get("id")}
    if py_ids and nb_ids and py_ids != nb_ids:
        return [_diag("exported-py-stale", "structure", rel, f"Generated `{py_path.name}` cell IDs differ from notebook exports", hint="Run `nbdev-export` to sync generated Python")]
    py_code_lines = sum(bool(ln.strip()) and not ln.lstrip().startswith("#") for ln in py_text.splitlines())
    if py_code_lines > 800:
        return [_diag("large-generated-py", "structure", rel, f"Generated `{py_path.name}` has {py_code_lines} code lines", hint="Split the notebook into smaller focused modules")]
    return []

def _structure_problems(nb_path: Path, nb, rel: str, ch_map: dict[int, str]) -> list[Diagnostic]:
    problems = _heading_problems(nb, rel, ch_map) + _export_sync_problems(nb_path, nb, rel)
    if len(nb.cells) > 120:
        problems.append(_diag("large-notebook", "structure", rel, f"Notebook has {len(nb.cells)} cells", hint="Split into focused notebooks so each tells one story"))
    seen_imports = {}
    for idx, cell in enumerate(nb.cells):
        if cell.get("cell_type") != "code": continue
        ch = ch_map.get(idx, "")
        tree, err = parse_cell_ast(cell)
        if err is not None:
            problems.append(_diag("invalid-python-cell", "structure", rel, f"Python syntax error: {err.msg}", cell=cell, chapter=ch, lineno=err.lineno or 0, severity="error"))
            continue
        if tree is None or not tree.body: continue
        has_imp = any(_is_import_stmt(n) for n in tree.body)
        has_other = any(not _is_import_stmt(n) for n in tree.body)
        if has_imp and has_other and not is_eval_false(cell):
            problems.append(_diag("mixed-import-cell", "structure", rel, "Cell mixes top-level imports with other statements", cell=cell, chapter=ch, hint="Put imports in their own dedicated cell"))
        scope = "exported" if is_exported(cell) else "lesson"
        for imp in cell_imports(cell):
            key = (scope, imp.module, imp.name, imp.alias)
            prev_cid = seen_imports.get(key)
            if prev_cid and prev_cid != str(cell.get("id", "")):
                problems.append(_diag("duplicate-import", "structure", rel, f"Duplicate {scope} import `{imp.alias}` (already imported in cell {prev_cid})", cell=cell, chapter=ch, lineno=imp.lineno, hint="Keep one shared import cell per scope"))
            else: seen_imports[key] = str(cell.get("id", ""))
    return problems

Here a small notebook with a heading jump, mixed imports, and a syntax error triggers the exact structural diagnostics.

In [ ]:
with TemporaryDirectory() as tmp:
    p = Path(tmp) / "bad_struct.ipynb"
    write_nb(new_nb([mk_cell("# Title\nBody in heading cell", "markdown"), mk_cell("### Skipped H2", "markdown"),
        mk_cell("import os\nx = 1"), mk_cell("def broken(:")]), p)
    nb = read_nb(p)
    codes = {d.code for d in _structure_problems(p, nb, p.name, _chapter_map(nb))}
    test_eq({"heading-not-standalone", "heading-level-jump", "mixed-import-cell", "invalid-python-cell"} <= codes, True)
sorted(codes)

## Execution order and state checks

Notebooks fail in subtle ways when a cell calls something defined further down, reads a variable from an `#| eval: false` cell, silently overwrites a shared name used earlier, or keeps a saved error traceback.

In [ ]:
#| export
_BUILTINS = frozenset(dir(builtins))
_SHORT_SCRATCH = frozenset({"nb", "df", "ax", "fn", "ch", "st", "ts", "ok", "res", "out", "tmp", "val", "err", "msg", "src", "txt", "raw", "idx", "row", "col", "key", "cfg", "doc", "ret", "ans"})

def _top_level_calls(tree: ast.Module) -> list[tuple[str, int]]:
    calls = []
    def walk(node):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef, ast.Lambda)): return
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name): calls.append((node.func.id, node.lineno))
        for child in ast.iter_child_nodes(node): walk(child)
    for stmt in tree.body: walk(stmt)
    return calls

def _later_binder(bindings_by_idx: list, name: str, after_idx: int) -> int | None:
    for j in range(after_idx + 1, len(bindings_by_idx)):
        if name in bindings_by_idx[j]: return j
    return None

`_state_problems` scans code cells top to bottom, tracking active bindings, `#| eval: false`-only definitions, and read-then-reassigned variables.

In [ ]:
#| export
def _state_problems(nb, rel: str, ch_map: dict[int, str]) -> list[Diagnostic]:
    problems = []
    defs_per_cell, reads_per_cell, eval_flags = [], [], []
    for cell in nb.cells:
        if cell.get("cell_type") == "code":
            d, r, _ = cell_dependencies(cell)
            defs_per_cell.append(set(d))
            reads_per_cell.append(set(r))
            eval_flags.append(is_eval_false(cell))
        else:
            defs_per_cell.append(set())
            reads_per_cell.append(set())
            eval_flags.append(False)
    active_defs, eval_only_defs, star_imported = set(_BUILTINS), set(), False
    defined_in = {}
    read_after_def = set()
    for idx, cell in enumerate(nb.cells):
        if cell.get("cell_type") != "code": continue
        ch = ch_map.get(idx, "")
        if any(o.get("output_type") == "error" for o in (cell.get("outputs") or [])):
            problems.append(_diag("error-output-saved", "state", rel, "Cell stores a saved error traceback in its outputs", cell=cell, chapter=ch, severity="error", hint="Fix the error and re-run the notebook"))
        if eval_flags[idx]:
            eval_only_defs.update(defs_per_cell[idx] - active_defs)
            continue
        for name in sorted(reads_per_cell[idx] & eval_only_defs - active_defs):
            problems.append(_diag("eval-false-dependency", "state", rel, f"Active cell depends on `{name}` defined only in an `#| eval: false` cell", cell=cell, chapter=ch, severity="error", hint="Define the dependency in an executed cell or mark this cell `#| eval: false`"))
        tree, _ = parse_cell_ast(cell)
        cell_avail = active_defs | defs_per_cell[idx]
        if tree is not None and not star_imported:
            for cname, lineno in _top_level_calls(tree):
                if cname in cell_avail: continue
                later = _later_binder(defs_per_cell, cname, idx)
                if later is not None:
                    cid_later = nb.cells[later].get("id", later)
                    problems.append(_diag("cell-order", "state", rel, f"`{cname}` is called before its definition/import in later cell {cid_later}", cell=cell, chapter=ch, lineno=lineno, severity="error", hint="Move the definition or import above this call"))
                else:
                    problems.append(_diag("missing-import", "state", rel, f"`{cname}` is called without an earlier definition or import", cell=cell, chapter=ch, lineno=lineno, hint="Import or define the callable above this cell"))
        if not is_exported(cell):
            for name in reads_per_cell[idx]:
                if name in defined_in: read_after_def.add(name)
            for name in sorted(defs_per_cell[idx]):
                if name in read_after_def and len(name) >= 4 and not name.startswith("_") and name not in _SHORT_SCRATCH:
                    problems.append(_diag("shared-name-reassigned", "state", rel, f"Shared variable `{name}` was read earlier and is reassigned here", cell=cell, chapter=ch, hint="Use a distinct name so re-running earlier cells stays deterministic"))
                elif name not in defined_in: defined_in[name] = idx
        if any(i.is_star for i in cell_imports(cell)): star_imported = True
        active_defs.update(defs_per_cell[idx])
        eval_only_defs.difference_update(defs_per_cell[idx])
    return problems

A four-cell sequence exercises cell-order failure, `#| eval: false` dependency, shared-name reassignment, and a saved error output.

In [ ]:
nb_state_demo = new_nb([
    mk_cell("result = later_fn(1)"),
    mk_cell("#| eval: false\nsecret_cfg = 9"),
    mk_cell("def later_fn(x): return x + secret_cfg"),
    mk_cell("shared_total = 10"),
    mk_cell("assert shared_total == 10"),
    mk_cell("shared_total = 20", outputs=[dict(output_type="error", ename="ValueError", evalue="boom")]),
])
state_codes = {d.code for d in _state_problems(nb_state_demo, "demo.ipynb", _chapter_map(nb_state_demo))}
test_eq({"cell-order", "eval-false-dependency", "shared-name-reassigned", "error-output-saved"} <= state_codes, True)
sorted(state_codes)

## Narrative and human-understanding checks

A green style checker means nothing if half the notebook is hidden behind `#| hide` or if examples live 100 cells away from the code they teach. These checks enforce honest visibility, chapter-local lessons, and idiomatic nbdev examples.

In [ ]:
#| export
_ASSERT_FUNCS = frozenset({"test_eq", "test_ne", "test_fail", "expect_fail", "test_close", "test_is", "test_stdout", "test_warns", "test_shuffled", "test"})

def _has_assertion(tree: ast.Module) -> bool:
    for node in ast.walk(tree):
        if isinstance(node, ast.Assert): return True
        if isinstance(node, ast.Call):
            fn = node.func
            name = fn.id if isinstance(fn, ast.Name) else fn.attr if isinstance(fn, ast.Attribute) else ""
            if name in _ASSERT_FUNCS: return True
    return False

def _cell_refs_any(tree: ast.Module, names: set[str]) -> bool:
    for node in ast.walk(tree):
        if isinstance(node, ast.Name) and node.id in names: return True
        if isinstance(node, ast.Attribute) and node.attr in names: return True
    return False

def _example_antipatterns(cell, rel: str, ch: str) -> list[Diagnostic]:
    tree, _ = parse_cell_ast(cell)
    if tree is None or not tree.body: return []
    found = []
    last = tree.body[-1]
    if isinstance(last, ast.Expr) and isinstance(last.value, ast.Call) and isinstance(last.value.func, ast.Name) and last.value.func.id == "print":
        found.append(_diag("example-trailing-print", "narrative", rel, "Example cell ends with `print(...)` instead of displaying a value", cell=cell, chapter=ch, lineno=last.lineno, hint="End the cell with the bare expression so Jupyter renders its rich display"))
    if any(isinstance(stmt, ast.Try) for stmt in tree.body):
        found.append(_diag("example-try-except", "narrative", rel, "Example cell uses top-level `try`/`except`", cell=cell, chapter=ch, hint="Use `expect_fail` or `test_fail` to demonstrate expected errors concisely"))
    return found

Chapter literacy checks look for explanation, visible example, and assertion **inside the chapter that exports the behavior** — distinguishing a missing example from a distant one in another chapter — while guarding against hidden-code walls.

In [ ]:
#| export
def _hide_balance_problems(nb, rel: str, ch_map: dict[int, str]) -> list[Diagnostic]:
    problems = []
    code_cells = [(i, c) for i, c in enumerate(nb.cells) if c.get("cell_type") == "code"]
    hidden_count = sum(is_hidden(c) for _, c in code_cells)
    if len(code_cells) >= 4 and (hidden_count / len(code_cells)) > 0.35:
        pct = round(100 * hidden_count / len(code_cells))
        problems.append(_diag("excessive-hidden-code", "narrative", rel, f"{pct}% of code cells ({hidden_count}/{len(code_cells)}) use `#| hide`", hint="Keep implementation visible and explained instead of hiding large sections behind `#| hide`"))
    run = []
    for idx, cell in enumerate(nb.cells):
        if cell.get("cell_type") == "code" and is_hidden(cell): run.append((idx, cell))
        else:
            if len(run) >= 4:
                problems.append(_diag("hidden-implementation-run", "narrative", rel, f"{len(run)} consecutive `#| hide` code cells with no visible narrative", cell=run[0][1], chapter=ch_map.get(run[0][0], ""), hint="Expose key helpers in narrative order or simplify the module"))
            run = []
    if len(run) >= 4:
        problems.append(_diag("hidden-implementation-run", "narrative", rel, f"{len(run)} consecutive `#| hide` code cells with no visible narrative", cell=run[0][1], chapter=ch_map.get(run[0][0], ""), hint="Expose key helpers in narrative order or simplify the module"))
    return problems

def _narrative_problems(nb, rel: str, ch_map: dict[int, str]) -> list[Diagnostic]:
    problems = _hide_balance_problems(nb, rel, ch_map)
    chaps = chapters(nb)
    other_refs = {}
    for ch in chaps:
        for cell in ch.cells:
            if cell.get("cell_type") != "code" or is_exported(cell): continue
            tree, _ = parse_cell_ast(cell)
            if tree is None: continue
            if not is_hidden(cell): problems.extend(_example_antipatterns(cell, rel, ch.title))
            names = {n.id for n in ast.walk(tree) if isinstance(n, ast.Name)} | {n.attr for n in ast.walk(tree) if isinstance(n, ast.Attribute)}
            for nm in names: other_refs.setdefault(nm, set()).add(ch.title)
    for ch in chaps:
        pub_fns = [d.name for c in ch.cells if is_exported(c) for d in cell_defs(c) if d.is_public and d.kind != "class"]
        pub_all = set(ch.defs)
        if not pub_all: continue
        has_expl = any(c.get("cell_type") == "markdown" and not heading_info(c) and cell_source(c).strip() for c in ch.cells)
        if not has_expl:
            problems.append(_diag("chapter-missing-explanation", "narrative", rel, f"Chapter `{ch.title}` exports {sorted(pub_all)} without a Markdown explanation", cell=ch.cells[0], chapter=ch.title, hint="Add a concise explanation stating what the chapter solves and its contract"))
        lesson_trees = [t for c in ch.cells if c.get("cell_type") == "code" and not is_exported(c) for t, _ in [parse_cell_ast(c)] if t is not None]
        vis_trees = [t for c in ch.cells if c.get("cell_type") == "code" and not is_exported(c) and not is_hidden(c) for t, _ in [parse_cell_ast(c)] if t is not None]
        if pub_fns:
            fn_set = set(pub_fns)
            if not any(_cell_refs_any(t, fn_set) for t in vis_trees):
                elsewhere = sorted({t for nm in fn_set for t in other_refs.get(nm, set()) if t != ch.title})
                if elsewhere:
                    problems.append(_diag("distant-example", "narrative", rel, f"Chapter `{ch.title}` has no local example for {pub_fns} (referenced only in {elsewhere})", cell=ch.cells[0], chapter=ch.title, hint="Place a compact visible example beside the implementation in this chapter"))
                else:
                    problems.append(_diag("chapter-missing-example", "narrative", rel, f"Chapter `{ch.title}` exports {pub_fns} without a visible example", cell=ch.cells[0], chapter=ch.title, hint="Add a small visible example demonstrating the chapter's behavior"))
            if not any(_has_assertion(t) for t in lesson_trees):
                problems.append(_diag("chapter-missing-assertion", "narrative", rel, f"Chapter `{ch.title}` exports {pub_fns} without an assertion", cell=ch.cells[0], chapter=ch.title, hint="Add a focused assertion checking the chapter's central guarantee"))
    return problems

This synthetic notebook hides 4 of 5 code cells, lacks a local explanation and assertion in its chapter, and ends an example with `print`.

In [ ]:
nb_narr = new_nb([
    mk_cell("## Hidden wall", "markdown"),
    mk_cell("#| export\ndef greet(name): return f'hi {name}'"),
    mk_cell("#| hide\na = 1"), mk_cell("#| hide\nb = 2"), mk_cell("#| hide\nc = 3"), mk_cell("#| hide\nd = 4"),
    mk_cell("print(greet('Ada'))"),
])
narr_codes = {d.code for d in _narrative_problems(nb_narr, "narr.ipynb", _chapter_map(nb_narr))}
test_eq({"excessive-hidden-code", "hidden-implementation-run", "chapter-missing-explanation", "chapter-missing-assertion", "example-trailing-print"} <= narr_codes, True)
sorted(narr_codes)

## Combining with `chkstyle`

`review_nb` runs our structural, state, and narrative checks and optionally merges `chkstyle` findings into the exact same `Diagnostic` format. `check_nb` prints the report and returns a shell-friendly exit code.

In [ ]:
#| export
_CELL_ID_TAG_RE = re.compile(r":cell\[([^\]]+)\]$")

def _split_hint(text: str) -> tuple[str, str]:
    if " (hint: " in text and text.endswith(")"):
        msg, hint = text.rsplit(" (hint: ", 1)
        return msg.strip(), hint[:-1].strip()
    return text.strip(), ""

def _run_chkstyle(nb_path: Path, rel: str, ch_map: dict[int, str], id_to_idx: dict[str, int]) -> list[Diagnostic]:
    try: from chkstyle import core as chk_core
    except ModuleNotFoundError: return []
    found = []
    for cell_path, lineno, rule, raw_msg, _ in chk_core.check_notebook(str(nb_path)):
        m = _CELL_ID_TAG_RE.search(cell_path)
        cid = m.group(1) if m else ""
        msg, hint = _split_hint(raw_msg)
        ch = ch_map.get(id_to_idx.get(cid, -1), "")
        found.append(Diagnostic(rule, "style", "warning", rel, msg, cid, ch, int(lineno or 0), hint, "chkstyle"))
    return found

`review_nb` aggregates all four categories per notebook, and `check_nb` prints the report and returns an exit status for scripts and CI.

In [ ]:
#| export
def review_nb(
    path: str | Path = ".",  # Notebook or directory to review
    *,
    include_style: bool = True,  # Include `chkstyle` code and baseline narrative findings
) -> ReviewReport:
    "Run structural, state, narrative, and optional `chkstyle` diagnostics on notebooks under `path`."
    p = Path(path).expanduser()
    root = project_root(p)
    diags, metrics = [], {}
    for nb_path in notebook_paths(p):
        nb = read_nb(nb_path)
        rel = _rel(nb_path, root)
        ch_map = _chapter_map(nb)
        id_to_idx = {str(c.get("id", "")): i for i, c in enumerate(nb.cells)}
        diags.extend(_structure_problems(nb_path, nb, rel, ch_map))
        diags.extend(_state_problems(nb, rel, ch_map))
        diags.extend(_narrative_problems(nb, rel, ch_map))
        if include_style: diags.extend(_run_chkstyle(nb_path, rel, ch_map, id_to_idx))
        code_n = sum(c.get("cell_type") == "code" for c in nb.cells)
        hide_n = sum(c.get("cell_type") == "code" and is_hidden(c) for c in nb.cells)
        metrics[rel] = dict(cells=len(nb.cells), code=code_n, hidden=hide_n, hidden_pct=round(100 * hide_n / code_n) if code_n else 0)
    return ReviewReport(str(p), diags, metrics)

def check_nb(
    path: str | Path = ".",  # Notebook or directory to check
    *,
    include_style: bool = True,  # Include `chkstyle` findings
    strict: bool = False,  # Raise `RuntimeError` if any finding exists
) -> int:
    "Print a compact review report and return `0` when clean or `1` when findings exist."
    report = review_nb(path, include_style=include_style)
    print(report)
    if strict and report.diagnostics: raise RuntimeError(str(report))
    return 1 if report.diagnostics else 0

A clean, literate notebook passes `review_nb` with zero findings across every category.

In [ ]:
with TemporaryDirectory() as tmp:
    p = Path(tmp) / "good.ipynb"
    write_nb(new_nb([
        mk_cell("# Good example", "markdown"), mk_cell("> Clean nbdev lesson.", "markdown"),
        mk_cell("## Double", "markdown"), mk_cell("`double` multiplies a number by two.", "markdown"),
        mk_cell("#| export\ndef double(x): return x * 2"),
        mk_cell("val = double(21)\nassert val == 42\nval"),
    ]), p)
    rep = review_nb(p, include_style=True)
    test_eq((rep.ok, rep.counts["total"]), (True, 0))
rep